In [1]:
import numpy as np
import pandas as pd
from itertools import product

def concentration_score(file_R0='R0.csv', 
                        file_sigma='sigma.csv',
                        true_R0=None,
                        true_sigma=None,
                        selected_indices=None):

    data_R0 = pd.read_csv(file_R0, header=None).values.ravel()
    data_sigma = pd.read_csv(file_sigma, header=None).values.ravel()
    
    # print(f"Loaded {len(data_R0)} {len(data_sigma)} samples")
    selected_R0 = data_R0[selected_indices]
    selected_sigma = data_sigma[selected_indices]
    
    d_R0    = ((selected_R0 - true_R0)    ** 2).mean()
    d_sigma = ((selected_sigma - true_sigma) ** 2).mean()
    return d_R0 + d_sigma

def compute_distance(filepath: str,
    standard_point: tuple,
    weights: tuple = None,
    metric: str = "euclidean",  # "euclidean", "manhattan", "chebyshev", "minkowski", "cosine"
    p: float = 3,               # only used when metric="minkowski"
    ):
                     
     # 1. Load
    df = pd.read_csv(filepath, header=None, names=["A", "B", "C", "D"])

    # 2. Min-Max normalization
    col_min = df.min()
    col_max = df.max()
    df_norm = (df - col_min) / (col_max - col_min)

    # 3. Normalize the standard point on the same scale
    standard = np.array(standard_point)
    standard_norm = (standard - col_min.values) / (col_max.values - col_min.values)

    # 4. Resolve weights (normalize so they sum to 1)
    if weights is not None:
        w = np.array(weights, dtype=float)
        if len(w) != 4:
            raise ValueError("weights must have exactly 4 values (wA, wB, wC, wD).")
        if np.any(w < 0):
            raise ValueError("All weights must be non-negative.")
        w = w / w.sum()          # normalize to sum = 1
    else:
        w = np.array([0.25, 0.25, 0.25, 0.25])   # equal weights

    # 5. Weighted different distance functions: Euclidean, manhattan, chebyshev, minkowski, cosine
    diff = (df_norm[["A", "B", "C", "D"]] - standard_norm).values
    if metric == "euclidean":
        dist = np.sqrt((w * diff ** 2).sum(axis=1))

    elif metric == "manhattan":
        dist = (w * np.abs(diff)).sum(axis=1)

    elif metric == "chebyshev":
        dist = (w * np.abs(diff)).max(axis=1)

    elif metric == "minkowski":
        dist = ((w * np.abs(diff) ** p).sum(axis=1)) ** (1 / p)

    elif metric == "cosine":
        dot     = (w * df_norm[["A", "B", "C", "D"]].values * standard_norm).sum(axis=1)
        norm_x  = np.sqrt((w * df_norm[["A", "B", "C", "D"]].values ** 2).sum(axis=1))
        norm_x0 = np.sqrt((w * standard_norm ** 2).sum())
        dist    = 1 - dot / (norm_x * norm_x0)

    else:
        raise ValueError(f"Unknown metric '{metric}'. Choose from: euclidean, manhattan, chebyshev, minkowski, cosine.")

    df_norm["distance"] = dist

    return df_norm["distance"]

In [2]:
##### setting: R0=2.0, sigma=0.8

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 2.0, 0.8   # ← your true values
percentile = 0.05
standard_point=(22.56521739, 17.67850798, -0.40284759,  6.6548344)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R02p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R02p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R02p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                   weights     score     n
43   (0.1, 0.1, 0.5, 0.8)  0.162993  4000
34   (0.1, 0.1, 0.4, 0.8)  0.164247  4000
42   (0.1, 0.1, 0.5, 0.7)  0.164762  4000
78   (0.1, 0.1, 0.9, 0.7)  0.164788  4000
116  (0.1, 0.2, 0.4, 0.9)  0.164855  4000
79   (0.1, 0.1, 0.9, 0.8)  0.164952  4000
71   (0.1, 0.1, 0.8, 0.9)  0.165148  4000
44   (0.1, 0.1, 0.5, 0.9)  0.165263  4000
782  (0.2, 0.1, 0.6, 0.9)  0.165803  4000
33   (0.1, 0.1, 0.4, 0.7)  0.165843  4000
773  (0.2, 0.1, 0.5, 0.9)  0.166053  4000
26   (0.1, 0.1, 0.3, 0.9)  0.166169  4000
35   (0.1, 0.1, 0.4, 0.9)  0.166889  4000
69   (0.1, 0.1, 0.8, 0.7)  0.167614  4000
52   (0.1, 0.1, 0.6, 0.8)  0.167623  4000
24   (0.1, 0.1, 0.3, 0.7)  0.167689  4000
781  (0.2, 0.1, 0.6, 0.8)  0.167717  4000
77   (0.1, 0.1, 0.9, 0.6)  0.167931  4000
70   (0.1, 0.1, 0.8, 0.8)  0.167948  4000
772  (0.2, 0.1, 0.5, 0.8)  0.168022  4000
manhattan distance: 
                    weights     score     n
223   (0.1, 0.3, 0.7, 0.8)  0.1

In [4]:
##### setting: R0=2.5, sigma=0.8

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 2.5, 0.8   # ← your true values
percentile = 0.05
standard_point=(26.65217391, 26.81859717, -0.50242938,  6.56234257)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R02p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R02p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R02p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                   weights     score     n
62   (0.1, 0.1, 0.7, 0.9)  0.495562  4000
44   (0.1, 0.1, 0.5, 0.9)  0.499405  4000
43   (0.1, 0.1, 0.5, 0.8)  0.499466  4000
42   (0.1, 0.1, 0.5, 0.7)  0.500496  4000
52   (0.1, 0.1, 0.6, 0.8)  0.501039  4000
772  (0.2, 0.1, 0.5, 0.8)  0.503050  4000
53   (0.1, 0.1, 0.6, 0.9)  0.503301  4000
809  (0.2, 0.1, 0.9, 0.9)  0.503903  4000
773  (0.2, 0.1, 0.5, 0.9)  0.504740  4000
70   (0.1, 0.1, 0.8, 0.8)  0.506268  4000
799  (0.2, 0.1, 0.8, 0.8)  0.507516  4000
32   (0.1, 0.1, 0.4, 0.6)  0.507595  4000
71   (0.1, 0.1, 0.8, 0.9)  0.507690  4000
60   (0.1, 0.1, 0.7, 0.7)  0.507778  4000
51   (0.1, 0.1, 0.6, 0.7)  0.507987  4000
80   (0.1, 0.1, 0.9, 0.9)  0.508759  4000
808  (0.2, 0.1, 0.9, 0.8)  0.509169  4000
782  (0.2, 0.1, 0.6, 0.9)  0.509221  4000
35   (0.1, 0.1, 0.4, 0.9)  0.509772  4000
790  (0.2, 0.1, 0.7, 0.8)  0.510065  4000
manhattan distance: 
                    weights     score     n
56    (0.1, 0.1, 0.7, 0.3)  0.5

In [5]:
##### setting: R0=3.0, sigma=0.8

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 3.0, 0.8   # ← your true values
percentile = 0.05
standard_point=(36.2173913,  29.28225731, -0.34643714,  6.08328067)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R03p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R03p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R03p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                   weights     score     n
397  (0.1, 0.5, 0.9, 0.2)  0.718732  4000
388  (0.1, 0.5, 0.8, 0.2)  0.722894  4000
479  (0.1, 0.6, 0.9, 0.3)  0.726768  4000
379  (0.1, 0.5, 0.7, 0.2)  0.728858  4000
560  (0.1, 0.7, 0.9, 0.3)  0.737260  4000
558  (0.1, 0.7, 0.9, 0.1)  0.738106  4000
470  (0.1, 0.6, 0.8, 0.3)  0.738707  4000
478  (0.1, 0.6, 0.9, 0.2)  0.739813  4000
639  (0.1, 0.8, 0.9, 0.1)  0.740396  4000
468  (0.1, 0.6, 0.8, 0.1)  0.741241  4000
289  (0.1, 0.4, 0.6, 0.2)  0.746720  4000
416  (0.1, 0.6, 0.2, 0.3)  0.747071  4000
561  (0.1, 0.7, 0.9, 0.4)  0.748819  4000
398  (0.1, 0.5, 0.9, 0.3)  0.749581  4000
630  (0.1, 0.8, 0.8, 0.1)  0.749598  4000
469  (0.1, 0.6, 0.8, 0.2)  0.750304  4000
316  (0.1, 0.4, 0.9, 0.2)  0.750838  4000
389  (0.1, 0.5, 0.8, 0.3)  0.750883  4000
477  (0.1, 0.6, 0.9, 0.1)  0.750968  4000
641  (0.1, 0.8, 0.9, 0.3)  0.751031  4000
manhattan distance: 
                   weights     score     n
317  (0.1, 0.4, 0.9, 0.3)  0.737

In [ ]:
##### setting: R0=3.5, sigma=0.8

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 3.5, 0.8   # ← your true values
percentile = 0.05
standard_point=(56.2173913,  35.24325406, -0.48419713, 10.39730631)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R03p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R03p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R03p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

In [6]:
##### setting: R0=4.0, sigma=0.8

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 4.0, 0.8   # ← your true values
percentile = 0.05
standard_point=(67.82608696, 41.68240546, -0.45651737, 10.29495928)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R04p0.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R04p0.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R04p0.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
2511  (0.4, 0.5, 0.1, 0.1)  1.085770  4000
3240  (0.5, 0.5, 0.1, 0.1)  1.091943  4000
5032  (0.7, 0.9, 0.2, 0.2)  1.094028  4000
5589  (0.8, 0.7, 0.1, 0.1)  1.096339  4000
3888  (0.6, 0.4, 0.1, 0.1)  1.096769  4000
5761  (0.8, 0.9, 0.2, 0.2)  1.096784  4000
4050  (0.6, 0.6, 0.1, 0.1)  1.098545  4000
4792  (0.7, 0.6, 0.2, 0.5)  1.101932  4000
3402  (0.5, 0.7, 0.1, 0.1)  1.102023  4000
1782  (0.3, 0.5, 0.1, 0.1)  1.102913  4000
6409  (0.9, 0.8, 0.2, 0.2)  1.103454  4000
4860  (0.7, 0.7, 0.1, 0.1)  1.106003  4000
5603  (0.8, 0.7, 0.2, 0.6)  1.106720  4000
4303  (0.6, 0.9, 0.2, 0.2)  1.107187  4000
6328  (0.9, 0.7, 0.2, 0.2)  1.107681  4000
4779  (0.7, 0.6, 0.1, 0.1)  1.107921  4000
4063  (0.6, 0.6, 0.2, 0.5)  1.107989  4000
6490  (0.9, 0.9, 0.2, 0.2)  1.108637  4000
3421  (0.5, 0.7, 0.3, 0.2)  1.109063  4000
3502  (0.5, 0.8, 0.3, 0.2)  1.109256  4000
manhattan distance: 
                    weights     score     n
5761  (0.8

In [3]:
##### setting: R0=1.5, sigma=0.8

stats = ["avg_prev", "div_prev", "npmi", "div_all_isolates"]
true_R0, true_sigma = 1.5, 0.8   # ← your true values
percentile = 0.05
standard_point=(8.04347826,  5.93509026, -0.41393641,  3.92980836)

# Grid over weights
weight_values = [i/10 for i in range(1, 10)]  # 0.1, 0.2, ..., 0.9

# files
file_ss="../../experimental_data/from_260312/ss_2params_R01p5.csv"
file_R0='../../experimental_data/from_260312/R0_samps_2params_R01p5.csv'
file_sigma='../../experimental_data/from_260312/sigma_samps_2params_R01p5.csv'


    # 1. Euclidean disntance for different weight vectors


results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="euclidean")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("euclidean distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    # 2. Manhattan distance for different weight vectors

# Grid over weights
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="manhattan")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("manhattan distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #3. Chebyshev distance for different weight vectors

results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="chebyshev")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("chebyshev distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #4. Minkowski distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="minkowski")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("minkowski distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]


    #5. Cosine distance for different vectors
results = []

for w in product(weight_values, repeat=4):
    df_distance = compute_distance(filepath=file_ss,
                                   standard_point=standard_point,
                                   weights=w,            # ← will be normalized to sum=1
                                   metric="cosine")
    threshold = df_distance.quantile(percentile)
    selected_indices = df_distance <= threshold
    score = concentration_score(file_R0=file_R0,
                                file_sigma=file_sigma,
                                true_R0=true_R0,
                                true_sigma=true_sigma, 
                                selected_indices=selected_indices)
    results.append({"weights": w, "score": score, "n": len(selected_indices)})

results_df = pd.DataFrame(results).sort_values("score")
print("cosine distance: \n", results_df.head(20))
best_weights = results_df.iloc[0]["weights"]

euclidean distance: 
                    weights     score     n
3810  (0.6, 0.3, 0.1, 0.4)  0.106547  4000
6087  (0.9, 0.4, 0.2, 0.4)  0.106597  4000
3891  (0.6, 0.4, 0.1, 0.4)  0.106823  4000
6335  (0.9, 0.7, 0.2, 0.9)  0.107276  4000
4548  (0.7, 0.3, 0.2, 0.4)  0.107344  4000
4783  (0.7, 0.6, 0.1, 0.5)  0.107516  4000
6006  (0.9, 0.3, 0.2, 0.4)  0.107646  4000
4457  (0.7, 0.2, 0.1, 0.3)  0.107673  4000
3243  (0.5, 0.5, 0.1, 0.4)  0.107778  4000
3406  (0.5, 0.7, 0.1, 0.5)  0.108083  4000
6161  (0.9, 0.5, 0.1, 0.6)  0.108206  4000
4749  (0.7, 0.5, 0.6, 0.7)  0.108229  4000
4550  (0.7, 0.3, 0.2, 0.6)  0.108313  4000
4539  (0.7, 0.3, 0.1, 0.4)  0.108340  4000
2269  (0.4, 0.2, 0.1, 0.2)  0.108422  4000
5358  (0.8, 0.4, 0.2, 0.4)  0.108422  4000
6098  (0.9, 0.4, 0.3, 0.6)  0.108471  4000
6252  (0.9, 0.6, 0.2, 0.7)  0.108503  4000
4701  (0.7, 0.5, 0.1, 0.4)  0.108541  4000
3162  (0.5, 0.4, 0.1, 0.4)  0.108617  4000
manhattan distance: 
                    weights     score     n
1985  (0.3